# RematSplit: Recompute or Transmit Activation Management for Memory Constrained Split Fine Tuning of LLMs

Runtime: **A100 GPU** (Colab Pro). Run cells top to bottom. Set `HF_TOKEN` in Cell 2 (Mistral-7B and LLaMA-2-7B are gated on Hugging Face).
Set `SMOKE_TEST = True` first for a 5 minute end to end check on any GPU.

Data (downloaded automatically): LongAlpaca-12k (Hugging Face `Yukang/LongAlpaca-12k`) and the Raca et al. 5G production dataset (`uccmisl/5Gdataset`).
Real parts: split LoRA fine-tuning, store / recompute / transmit / int8 activation policies, measured memory and time.
Emulated parts: device compute speed, power, and the wireless link (edit `DEVICES` in Cell 9 with datasheet or measured values).
Outputs: CSV results in `Downloads/RematSplit/results`, 800 dpi figures in `Downloads/RematSplit`.

## 1. Install

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.51.3", "peft==0.15.2", "datasets>=3.0", "accelerate>=1.0"], check=True)
subprocess.run("echo ttf-mscorefonts-installer msttcorefonts/accepted-mscorefonts-eula select true | debconf-set-selections && "
               "apt-get -qq install -y ttf-mscorefonts-installer > /dev/null 2>&1 && fc-cache -f > /dev/null 2>&1",
               shell=True)
print("Install finished. If transformers was upgraded, use Runtime > Restart session once, then run from Cell 2.")

## 2. Configuration

In [ ]:
import os, gc, json, math, time, random, copy, glob, subprocess, zipfile
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

SMOKE_TEST = False                       # True: tiny random model, short sequences, runs on any GPU/CPU in minutes
MODEL_ID   = "mistralai/Mistral-7B-v0.1" # or "meta-llama/Llama-2-7b-hf" (needs Meta approval on Hugging Face)
HF_TOKEN   = ""                          # paste your Hugging Face read token here

SPLIT_LAYER   = 8                        # decoder layers kept on the device side
LORA_R, LORA_ALPHA = 16, 32
LORA_TARGETS  = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
SEQ_GRID      = [1024, 2048, 3072, 4096, 6144, 8192, 12288, 16384, 24576, 32768]
MEM_BUDGETS_GB = [6, 8, 12, 16]          # total device memory available to the fine-tuning process
TRAIN_SEQ, TRAIN_STEPS, LR, EVAL_SAMPLES = 2048, 300, 2e-4, 20
L_FIG, M_FIG_GB, LINK_FIG = 8192, 8, 50  # memory tight operating point for energy, breakdown and ablation figures
L_ONLINE, M_ONLINE_GB, ONLINE_STEPS = 12288, 8, 2000
ONLINE_DEVICE, ONLINE_SCALE = "Smartphone", 50
UL_DL_RATIO   = 0.25                     # uplink capacity as a fraction of the traced downlink throughput
LINK_SCALES   = [1, 10, 50, 200]         # trace scaling for link sensitivity (1 = raw 5G trace, 200 = Gbps class link)
SERVER_CKPT   = True                     # checkpoint server layers so the Colab GPU can host the server side
DP_BINS       = 400
SEED          = 42
OUT_DIR       = "/content/Downloads/RematSplit" if os.path.isdir("/content") else os.path.abspath("Downloads/RematSplit")
RES_DIR       = os.path.join(OUT_DIR, "results")

if SMOKE_TEST:
    SPLIT_LAYER, SEQ_GRID, MEM_BUDGETS_GB = 2, [128, 256, 384, 512, 768, 1024], [0.05, 0.08, 0.12, 0.2]
    TRAIN_SEQ, TRAIN_STEPS, EVAL_SAMPLES = 256, 20, 4
    L_FIG, M_FIG_GB, L_ONLINE, M_ONLINE_GB, ONLINE_STEPS = 512, 0.08, 768, 0.08, 300

os.makedirs(RES_DIR, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if DEVICE == "cuda" else torch.float32
GB = 1024 ** 3

def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if DEVICE == "cuda": torch.cuda.manual_seed_all(s)
seed_all()

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)

GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"
print("Device:", GPU_NAME, "| dtype:", DTYPE, "| output:", OUT_DIR)

## 3. Model, LoRA and split point

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, LlamaConfig, LlamaForCausalLM
from peft import LoraConfig, get_peft_model

TOK_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" if SMOKE_TEST else MODEL_ID
tokenizer = AutoTokenizer.from_pretrained(TOK_ID)

def build_model():
    seed_all()
    if SMOKE_TEST:
        cfg = LlamaConfig(vocab_size=len(tokenizer), hidden_size=256, intermediate_size=688, num_hidden_layers=4,
                          num_attention_heads=4, num_key_value_heads=4, max_position_embeddings=4096)
        cfg._attn_implementation = "sdpa"
        m = LlamaForCausalLM(cfg).to(DTYPE)
    else:
        m = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, attn_implementation="sdpa")
    m.config.use_cache = False
    m = m.to(DEVICE)
    for p in m.parameters(): p.requires_grad_(False)
    lcfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.0, bias="none",
                      target_modules=LORA_TARGETS, task_type="CAUSAL_LM")
    return get_peft_model(m, lcfg)

model = build_model()
base = model.base_model.model
LAYERS = base.model.layers
N_LAYERS, HID = len(LAYERS), base.config.hidden_size
INTER = base.config.intermediate_size
HEAD_DIM = getattr(base.config, "head_dim", None) or HID // base.config.num_attention_heads
QUANT_LASTDIMS = {HID, INTER, HEAD_DIM, base.config.num_key_value_heads * HEAD_DIM}
K = SPLIT_LAYER
UNITS = [(kind, i) for i in range(K) for kind in ("attn", "mlp")]
INIT_LORA = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}
PARAM_PTRS = {p.untyped_storage().data_ptr() for p in model.parameters()}

def nbytes_params(mods, trainable_mult=3):
    tot = 0
    for mod in mods:
        for p in mod.parameters():
            tot += p.numel() * p.element_size()
            if p.requires_grad: tot += p.numel() * p.element_size() * trainable_mult   # gradient + two Adam moments
    return tot

DEV_WEIGHT_BYTES = nbytes_params([base.model.embed_tokens] + [LAYERS[i] for i in range(K)])
print(f"layers={N_LAYERS} hidden={HID} split={K} device weights={DEV_WEIGHT_BYTES/GB:.2f} GB "
      f"trainable={sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f} M")

## 4. LongAlpaca-12k data

In [ ]:
from datasets import load_dataset

raw = load_dataset("Yukang/LongAlpaca-12k", split="train")
BOS, EOS = tokenizer.bos_token_id, tokenizer.eos_token_id

def tok_fn(batch):
    prompts = [(ins or "") + ("\n" + inp if inp else "") + "\n\n### Response:\n"
               for ins, inp in zip(batch["instruction"], batch["input"])]
    p = tokenizer(prompts, add_special_tokens=False)["input_ids"]
    o = tokenizer(batch["output"], add_special_tokens=False)["input_ids"]
    return {"p_ids": [[BOS] + a for a in p], "o_ids": [b + [EOS] for b in o]}

tok = raw.map(tok_fn, batched=True, batch_size=64, remove_columns=raw.column_names)
lens = np.array([len(a) + len(b) for a, b in zip(tok["p_ids"], tok["o_ids"])])
N_HELD = 200
TRAIN_IDX = np.arange(len(tok) - N_HELD)
HELD_IDX = np.arange(len(tok) - N_HELD, len(tok))
P_IDS, O_IDS = tok["p_ids"], tok["o_ids"]

def get_sample(L, start, pool=TRAIN_IDX):
    ids, lab, j = [], [], start
    while len(ids) < L:
        i = int(pool[j % len(pool)]); j += 1
        p, o = P_IDS[i], O_IDS[i]
        ids += p + o; lab += [-100] * len(p) + o
    ids, lab = ids[-L:], lab[-L:]
    return (torch.tensor(ids, device=DEVICE).unsqueeze(0), torch.tensor(lab, device=DEVICE).unsqueeze(0))

pd.DataFrame({"tokens": lens}).describe(percentiles=[.25, .5, .75, .9, .99]).to_csv(os.path.join(RES_DIR, "dataset_length_stats.csv"))
print(f"LongAlpaca-12k: {len(tok)} samples | median {int(np.median(lens))} tokens | "
      f">=8k: {(lens >= 8192).mean()*100:.1f}% | >=16k: {(lens >= 16384).mean()*100:.1f}%")

## 5. 5G link traces

In [ ]:
TRACE_ROOT = os.path.join(os.path.dirname(OUT_DIR), "5Gdataset")
if not os.path.isdir(TRACE_ROOT):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/uccmisl/5Gdataset.git", TRACE_ROOT], check=True)
TRACE_DIR = os.path.join(TRACE_ROOT, "unzipped")
if not os.path.isdir(TRACE_DIR):
    with zipfile.ZipFile(os.path.join(TRACE_ROOT, "5G-production-dataset.zip")) as z: z.extractall(TRACE_DIR)

frames = []
for f in sorted(glob.glob(os.path.join(TRACE_DIR, "5G-production-dataset", "Download", "*", "*.csv"))):
    d = pd.read_csv(f)
    d = pd.DataFrame({"dl_mbps": pd.to_numeric(d["DL_bitrate"], errors="coerce") / 1000.0,
                      "rtt_ms": pd.to_numeric(d["PINGAVG"], errors="coerce"),
                      "mobility": os.path.basename(os.path.dirname(f))}).dropna(subset=["dl_mbps"])
    frames.append(d)
TRACE = pd.concat(frames, ignore_index=True)
RTT_MED = float(TRACE["rtt_ms"].median())
TRACE["rtt_ms"] = TRACE["rtt_ms"].ffill().fillna(RTT_MED)
TRACE["dl_mbps"] = TRACE["dl_mbps"].clip(lower=0.5)
TRACE["ul_mbps"] = TRACE["dl_mbps"] * UL_DL_RATIO

def link_at(t, scale=1.0):
    r = TRACE.iloc[t % len(TRACE)]
    return dict(ul=r.ul_mbps * scale * 1e6 / 8, dl=r.dl_mbps * scale * 1e6 / 8, rtt=r.rtt_ms / 1000.0)

def link_static(scale=1.0, q=0.5):
    return dict(ul=TRACE.ul_mbps.quantile(q) * scale * 1e6 / 8, dl=TRACE.dl_mbps.quantile(q) * scale * 1e6 / 8,
                rtt=RTT_MED / 1000.0)

TRACE.describe().to_csv(os.path.join(RES_DIR, "trace_stats.csv"))
print(f"5G trace: {len(TRACE)} one second samples | median DL {TRACE.dl_mbps.median():.1f} Mbps | "
      f"median UL (scaled) {TRACE.ul_mbps.median():.1f} Mbps | median RTT {RTT_MED:.0f} ms")

## 6. Split fine-tuning engine with per unit activation policies

In [ ]:
MODES = ["S", "R", "T", "Q", "TQ"]
MODE_NAMES = {"S": "Store", "R": "Recompute", "T": "Transmit", "Q": "Int8 store", "TQ": "Int8 transmit"}
SERVER_STORE = "cpu"   # remote activation store of the server side

def tbytes(t): return t.numel() * t.element_size()

def sync():
    if DEVICE == "cuda": torch.cuda.synchronize()

def now():
    sync(); return time.perf_counter()

def quant8(t):
    tf = t.float()
    s = tf.abs().amax(dim=-1, keepdim=True).clamp_min(1e-8) / 127.0
    return torch.round(tf / s).clamp_(-127, 127).to(torch.int8), s

class Acc:
    def __init__(self, shared=()):
        self.shared = {x.untyped_storage().data_ptr() for x in shared}
        self.dev, self.tx = 0, 0

class UnitHooks:
    """Saved tensor policy of one unit: keep (S), int8 keep (Q), send to server store (T), int8 send (TQ)."""
    def __init__(self, acc, mode):
        self.acc, self.mode, self.cache, self.keep = acc, mode, {}, []
    def pack(self, t):
        sp = t.untyped_storage().data_ptr()
        if sp in PARAM_PTRS or sp in self.acc.shared or t.numel() == 0:
            return ("raw", t, None, None, None)
        key = (sp, t.storage_offset(), tuple(t.shape), tuple(t.stride()), t.dtype)
        if key in self.cache: return self.cache[key]
        self.keep.append(t)
        quant = (self.mode in ("Q", "TQ") and t.is_floating_point() and t.dim() >= 2
                 and t.shape[-1] in QUANT_LASTDIMS and t.numel() >= 1024)
        if quant:
            q, s = quant8(t); payload = ["q", q, s, t.dtype, t.device]; nb = tbytes(q) + tbytes(s)
        else:
            payload = ["raw", t, None, t.dtype, t.device]; nb = tbytes(t)
        if self.mode in ("T", "TQ"):
            payload[1] = payload[1].to(SERVER_STORE)
            if payload[2] is not None: payload[2] = payload[2].to(SERVER_STORE)
            self.acc.tx += nb
        else:
            self.acc.dev += nb
        payload = tuple(payload); self.cache[key] = payload
        return payload
    def unpack(self, payload):
        kind, a, s, dt, dev = payload
        if kind == "q":
            return (a.to(dev).float() * s.to(dev)).to(dt)
        return a.to(dev) if a.device != dev else a
    def close(self):
        self.cache.clear(); self.keep.clear()

def attn_fn(layer, x, pe):
    h = layer.input_layernorm(x)
    a = layer.self_attn(hidden_states=h, position_embeddings=pe, attention_mask=None)[0]
    return x + a

def mlp_fn(layer, x):
    return x + layer.mlp(layer.post_attention_layernorm(x))

def unit_fn(kind, i, pe):
    layer = LAYERS[i]
    return (lambda x: attn_fn(layer, x, pe)) if kind == "attn" else (lambda x: mlp_fn(layer, x))

def run_unit(fn, x, mode, acc):
    if mode == "R":
        acc.dev += tbytes(x)
        return checkpoint(fn, x, use_reentrant=False)
    h = UnitHooks(acc, mode)
    with torch.autograd.graph.saved_tensors_hooks(h.pack, h.unpack):
        y = fn(x)
    h.close()
    return y

def embed(ids):
    with torch.no_grad():
        x = base.model.embed_tokens(ids)
        pos = torch.arange(ids.shape[1], device=ids.device).unsqueeze(0)
        pe = base.model.rotary_emb(x, pos)
    return x, pe

def device_forward(ids, plan):
    x, pe = embed(ids)
    acc = Acc(shared=pe)
    for (kind, i), m in zip(UNITS, plan):
        x = run_unit(unit_fn(kind, i, pe), x, m, acc)
    return x, pe, acc

def server_loss(h, labels, pe, chunk=2048):
    for i in range(K, N_LAYERS):
        layer = LAYERS[i]
        f = lambda z, l=layer: mlp_fn(l, attn_fn(l, z, pe))
        h = checkpoint(f, h, use_reentrant=False) if SERVER_CKPT else f(h)
    h, lab = h[:, :-1], labels[:, 1:]
    n_valid = (lab != -100).sum().clamp_min(1)
    def ce(hc, lc):
        logits = base.lm_head(base.model.norm(hc)).float()
        return F.cross_entropy(logits.reshape(-1, logits.size(-1)), lc.reshape(-1), ignore_index=-100, reduction="sum")
    tot = 0.0
    for s in range(0, h.shape[1], chunk):
        tot = tot + checkpoint(ce, h[:, s:s + chunk], lab[:, s:s + chunk], use_reentrant=False)
    return tot / n_valid

def split_step(ids, labels, plan, opt=None):
    t0 = now()
    smashed, pe, acc = device_forward(ids, plan)
    t1 = now()
    s_in = smashed.detach().requires_grad_(True)               # uplink: smashed activations
    loss = server_loss(s_in, labels, pe)
    loss.backward()
    g = s_in.grad.detach()                                     # downlink: activation gradient
    t2 = now()
    smashed.backward(g)
    t3 = now()
    if opt is not None:
        opt.step(); opt.zero_grad(set_to_none=True)
    return dict(loss=float(loss.detach()), dev_bytes=acc.dev, tx_bytes=acc.tx,
                t_dev_fwd=t1 - t0, t_srv=t2 - t1, t_dev_bwd=t3 - t2)

def trainable(): return [p for p in model.parameters() if p.requires_grad]

def reset_lora():
    with torch.no_grad():
        for n, p in model.named_parameters():
            if p.requires_grad: p.copy_(INIT_LORA[n]); p.grad = None
print("Engine ready. Modes:", MODE_NAMES)

## 7. Correctness check (loss and gradient equivalence of all policies)

In [ ]:
L_CHK = 512 if SMOKE_TEST else 2048
ids, lab = get_sample(L_CHK, 0)

with torch.no_grad():
    hf_loss = float(model(input_ids=ids, labels=lab).loss)
    smashed, pe, _ = device_forward(ids, ["S"] * len(UNITS))
    split_loss = float(server_loss(smashed, lab, pe))
print(f"Loss check | Hugging Face forward {hf_loss:.6f} | split forward {split_loss:.6f}")

def grads_for(plan):
    reset_lora()
    out = split_step(ids, lab, plan)
    g = torch.cat([p.grad.double().flatten() for p in trainable() if p.grad is not None])
    reset_lora()
    return g, out

rows, g_ref = [], None
for m in MODES:
    g, out = grads_for([m] * len(UNITS))
    if g_ref is None: g_ref = g
    rel = float((g - g_ref).norm() / g_ref.norm())
    cos = float(F.cosine_similarity(g, g_ref, dim=0))
    rows.append(dict(mode=MODE_NAMES[m], loss=out["loss"], grad_rel_err=rel, grad_cosine=cos,
                     device_act_MB=out["dev_bytes"] / 2**20, transmitted_MB=out["tx_bytes"] / 2**20))
CHECK = pd.DataFrame(rows)
CHECK.to_csv(os.path.join(RES_DIR, "gradient_equivalence.csv"), index=False)
print(CHECK.to_string(index=False))

## 8. Per unit profiling (memory, compute, transmit bytes) across sequence lengths

In [ ]:
REPS = 3

def mem_now(): return torch.cuda.memory_allocated() if DEVICE == "cuda" else 0
def reset_peak():
    if DEVICE == "cuda": torch.cuda.reset_peak_memory_stats()
def peak(): return torch.cuda.max_memory_allocated() if DEVICE == "cuda" else 0
def free():
    gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()

def profile_unit(kind, x0, pe, mode):
    fn = unit_fn(kind, 0, pe)
    rec = []
    for r in range(REPS + 1):
        model.zero_grad(set_to_none=True); free()
        x = x0.detach().clone().requires_grad_(True); xb = tbytes(x)
        acc = Acc(shared=pe)
        base_mem = mem_now(); reset_peak()
        t0 = now(); y = run_unit(fn, x, mode, acc); t1 = now()
        pf = peak() - base_mem
        g = torch.randn_like(y); reset_peak()
        t2 = now(); y.backward(g); t3 = now()
        pb = peak() - base_mem
        if DEVICE != "cuda": pf = pb = acc.dev + 4 * xb
        rec.append(dict(res=acc.dev, tx=acc.tx, fwd=t1 - t0, bwd=t3 - t2,
                        pf=pf + xb, pb=pb + (xb if mode == "R" else 0)))
        del y, g, x
    rec = rec[1:]
    out = {k: float(np.median([r[k] for r in rec])) for k in rec[0]}
    model.zero_grad(set_to_none=True); free()
    return out

def profile_server(ids, lab):
    with torch.no_grad():
        smashed, pe, _ = device_forward(ids, ["S"] * len(UNITS))
    ts = []
    for r in range(2):
        s_in = smashed.detach().clone().requires_grad_(True)
        t0 = now(); loss = server_loss(s_in, lab, pe); loss.backward(); t1 = now()
        ts.append(t1 - t0); del s_in, loss
        model.zero_grad(set_to_none=True); free()
    return ts[-1]

prof_rows, srv_rows = [], []
for L in SEQ_GRID:
    try:
        ids, lab = get_sample(L, 7)
        x_emb, pe = embed(ids)
        with torch.no_grad(): x_mlp = attn_fn(LAYERS[0], x_emb, pe)
        for kind, x0 in (("attn", x_emb), ("mlp", x_mlp)):
            for m in MODES:
                r = profile_unit(kind, x0, pe, m)
                prof_rows.append(dict(L=L, kind=kind, mode=m, **r))
        del x_emb, x_mlp
        try:
            srv_rows.append(dict(L=L, t_srv=profile_server(ids, lab)))
        except torch.cuda.OutOfMemoryError:
            srv_rows.append(dict(L=L, t_srv=np.nan))
        print(f"profiled L={L}")
    except torch.cuda.OutOfMemoryError:
        print(f"L={L}: out of memory on {GPU_NAME}, skipped")
    model.zero_grad(set_to_none=True); free()

PROF = pd.DataFrame(prof_rows)
SRV = pd.DataFrame(srv_rows)
ok = SRV.dropna()
if len(ok) >= 2 and SRV.t_srv.isna().any():
    A = np.stack([ok.L.values, ok.L.values.astype(float) ** 2], 1)
    coef = np.linalg.lstsq(A, ok.t_srv.values, rcond=None)[0]
    miss = SRV.t_srv.isna()
    SRV.loc[miss, "t_srv"] = coef[0] * SRV.L[miss] + coef[1] * SRV.L[miss].astype(float) ** 2
    SRV["t_srv_extrapolated"] = miss
PROF.to_csv(os.path.join(RES_DIR, "unit_profile.csv"), index=False)
SRV.to_csv(os.path.join(RES_DIR, "server_profile.csv"), index=False)
L_OK = sorted(set(PROF.L))
P = {(r.L, r.kind, r.mode): r for r in PROF.itertuples()}
show = PROF.assign(res_MB=PROF.res / 2**20, tx_MB=PROF.tx / 2**20, fwd_ms=PROF.fwd * 1e3, bwd_ms=PROF.bwd * 1e3)
print(show[["L", "kind", "mode", "res_MB", "tx_MB", "fwd_ms", "bwd_ms"]].round(2).to_string(index=False))

## 9. Cost model, DP planner, latency hiding and baselines

In [ ]:
GPU_PEAK_TFLOPS = {"H100": 756.0, "A100": 312.0, "L4": 121.0, "V100": 125.0, "T4": 65.0}
HOST_TFLOPS = next((v for k, v in GPU_PEAK_TFLOPS.items() if k in GPU_NAME), 312.0)

# Device profiles. Peak FP16 tensor throughput and power increments are ASSUMED values:
# replace them with your datasheet values or tegrastats / Android power-rail measurements before reporting.
DEVICES = {
    "Jetson Orin": dict(tflops=42.5, p_idle=8.0, d_comp=30.0, d_tx=2.5, d_rx=1.5),
    "Smartphone":  dict(tflops=4.0,  p_idle=1.0, d_comp=5.0,  d_tx=2.5, d_rx=1.5),
}
for d in DEVICES.values(): d["slow"] = HOST_TFLOPS / d["tflops"]
RUNTIME_OVERHEAD = (0.002 if SMOKE_TEST else 0.5) * GB
ELEM = torch.tensor([], dtype=DTYPE).element_size()
KINDS = [u[0] for u in UNITS]

def smashed_bytes(L): return L * HID * ELEM
def act_budget(L, M): return M - DEV_WEIGHT_BYTES - 2 * smashed_bytes(L) - RUNTIME_OVERHEAD
def t_server(L): return float(SRV.set_index("L").t_srv[L])

def unit_cost(L, kind, m, dev, link):
    r = P[(L, kind, m)]
    c = P[(L, kind, {"T": "S", "TQ": "Q"}.get(m, m))]
    t_comp = (c.fwd + c.bwd) * dev["slow"]
    t_rc = P[(L, kind, "S")].fwd * dev["slow"] if m == "R" else 0.0
    t_up, t_dn = r.tx / link["ul"], r.tx / link["dl"]
    t = t_comp + t_up + t_dn
    e = dev["p_idle"] * t + dev["d_comp"] * t_comp + dev["d_tx"] * t_up + dev["d_rx"] * t_dn
    return dict(time=t, energy=e, t_comp=t_comp, t_rc=t_rc, t_link=t_up + t_dn)

def fixed_cost(L, dev, link):
    sb = smashed_bytes(L)
    t_up, t_dn = sb / link["ul"], sb / link["dl"]
    window = t_up + link["rtt"] + t_server(L) + t_dn
    e = dev["p_idle"] * window + dev["d_tx"] * t_up + dev["d_rx"] * t_dn
    return dict(window=window, time=window, energy=e, t_link=t_up + t_dn, t_wait=link["rtt"] + t_server(L))

def plan_peak(plan, L, hidden=()):
    res = [P[(L, k, m)].res for k, m in zip(KINDS, plan)]
    pre = np.concatenate([[0.0], np.cumsum(res)])
    hs = {h: P[(L, KINDS[h], "S")].res for h in hidden}
    peaks = [pre[-1] + sum(hs.values())]
    for u, (k, m) in enumerate(zip(KINDS, plan)):
        peaks.append(pre[u] + P[(L, k, m)].pf)
        hb = sum(v for h, v in hs.items() if h < u)
        mm = "S" if u in hs else m
        peaks.append(pre[u] + hb + P[(L, k, mm)].pb)
    return max(peaks)

def feasible(plan, L, M, hidden=()):
    return plan_peak(plan, L, hidden) <= act_budget(L, M)

def choose_hidden(plan, L, M, dev, link):
    win, hidden, used = fixed_cost(L, dev, link)["window"], [], 0.0
    for u in reversed(range(len(plan))):
        if plan[u] != "R": continue
        t = unit_cost(L, KINDS[u], "R", dev, link)["t_rc"]
        if used + t <= win and feasible(plan, L, M, hidden + [u]):
            hidden.append(u); used += t
    return hidden, used

def evaluate(plan, L, M, dev, link, hide=True):
    if plan is None or not feasible(plan, L, M): return None
    hidden, h_t = choose_hidden(plan, L, M, dev, link) if hide else ([], 0.0)
    fx = fixed_cost(L, dev, link)
    cs = [unit_cost(L, k, m, dev, link) for k, m in zip(KINDS, plan)]
    t_comp = sum(c["t_comp"] for c in cs); t_rc = sum(c["t_rc"] for c in cs)
    t_link = sum(c["t_link"] for c in cs) + fx["t_link"]
    time_ = sum(c["time"] for c in cs) + fx["time"] - h_t
    energy = sum(c["energy"] for c in cs) + fx["energy"] - dev["p_idle"] * h_t
    return dict(time=time_, energy=energy, t_comp=t_comp - t_rc, t_recomp=t_rc - h_t, t_hidden=h_t,
                t_link=t_link, t_wait=max(fx["t_wait"] - h_t, 0.0), peak_GB=plan_peak(plan, L, hidden) / GB
                + (DEV_WEIGHT_BYTES + 2 * smashed_bytes(L) + RUNTIME_OVERHEAD) / GB,
                n_hidden=len(hidden), hidden=hidden)

def dp_plan(L, M, dev, link, modes=MODES, objective="energy", G=DP_BINS):
    """Exact multiple choice DP over the unit sequence (memory discretised upward, so every plan returned is feasible)."""
    Mact = act_budget(L, M)
    if Mact <= 0: return None
    q = Mact / G
    cost = np.full(G + 1, np.inf); cost[0] = 0.0
    back = []
    for u, k in enumerate(KINDS):
        new = np.full(G + 1, np.inf); ch = np.full(G + 1, -1); prev = np.full(G + 1, -1)
        states = np.arange(G + 1) * q
        for mi, m in enumerate(modes):
            if (L, k, m) not in P: continue
            r = int(math.ceil(P[(L, k, m)].res / q - 1e-9))
            if r > G: continue
            pk = max(P[(L, k, m)].pf, P[(L, k, m)].pb)
            c = unit_cost(L, k, m, dev, link)[objective]
            ok = np.isfinite(cost) & (states + pk <= Mact)
            src = np.arange(G + 1 - r)
            cand = np.where(ok[src], cost[src] + c, np.inf)
            tgt = src + r
            better = cand < new[tgt]
            new[tgt[better]] = cand[better]; ch[tgt[better]] = mi; prev[tgt[better]] = src[better]
        cost = new; back.append((ch, prev))
    if not np.isfinite(cost).any(): return None
    s = int(np.argmin(cost)); plan = []
    for ch, prev in reversed(back):
        plan.append(modes[ch[s]]); s = prev[s]
    return plan[::-1]

def greedy_plan(L, M):
    plan = []
    for u in range(len(UNITS)):
        trial = plan + ["S"] + ["R"] * (len(UNITS) - u - 1)
        plan.append("S" if feasible(trial, L, M) else "R")
    return plan if feasible(plan, L, M) else None

def uniform(m): return lambda L, M, dev, link: [m] * len(UNITS)

METHODS = {
    "Store (SplitLoRA)":   (uniform("S"), True),
    "Checkpoint all":      (uniform("R"), True),
    "Offload all":         (uniform("T"), True),
    "Greedy":              (lambda L, M, dev, link: greedy_plan(L, M), True),
    "RematSplit":          (lambda L, M, dev, link: dp_plan(L, M, dev, link), True),
}
ABLATIONS = {
    "RematSplit":            (lambda L, M, dev, link: dp_plan(L, M, dev, link), True),
    "w/o latency hiding":    (lambda L, M, dev, link: dp_plan(L, M, dev, link), False),
    "w/o transmit":          (lambda L, M, dev, link: dp_plan(L, M, dev, link, modes=["S", "R", "Q"]), True),
    "w/o int8":              (lambda L, M, dev, link: dp_plan(L, M, dev, link, modes=["S", "R", "T"]), True),
    "Store/Recompute only":  (lambda L, M, dev, link: dp_plan(L, M, dev, link, modes=["S", "R"]), True),
}

def run_method(fn_hide, L, M, dev, link):
    fn, hide = fn_hide
    plan = fn(L, M, dev, link)
    ev = evaluate(plan, L, M, dev, link, hide=hide)
    return plan, ev

print(f"Host GPU peak {HOST_TFLOPS} TFLOPS | slowdown factors:",
      {k: round(v['slow'], 1) for k, v in DEVICES.items()})

## 10. Experiment 1 and 2: max sequence length, energy and time across budgets, devices and links

In [ ]:
L_FIG = max([l for l in L_OK if l <= L_FIG])
json.dump(dict(L_FIG=L_FIG, M_FIG_GB=M_FIG_GB, LINK_FIG=LINK_FIG, MEM_BUDGETS_GB=MEM_BUDGETS_GB, N_UNITS=len(UNITS), SPLIT_LAYER=K,
               MODEL_ID=MODEL_ID, SMOKE_TEST=SMOKE_TEST), open(os.path.join(RES_DIR, "settings.json"), "w"))
rows = []
for dname, dev in DEVICES.items():
    for scale in LINK_SCALES:
        link = link_static(scale)
        for Mg in MEM_BUDGETS_GB:
            M = Mg * GB
            for L in L_OK:
                for meth, fh in {**METHODS, **{k: v for k, v in ABLATIONS.items() if k != "RematSplit"}}.items():
                    plan, ev = run_method(fh, L, M, dev, link)
                    row = dict(device=dname, link_scale=scale, M_GB=Mg, L=L, method=meth, feasible=ev is not None,
                               plan="".join(f"{m}," for m in plan).rstrip(",") if plan else "")
                    if ev: row.update({k: v for k, v in ev.items() if k != "hidden"})
                    rows.append(row)
SWEEP = pd.DataFrame(rows)
SWEEP.to_csv(os.path.join(RES_DIR, "plan_sweep.csv"), index=False)

MAXLEN = (SWEEP[SWEEP.feasible].groupby(["device", "link_scale", "M_GB", "method"]).L.max()
          .unstack("method").reindex(columns=list(METHODS) + [k for k in ABLATIONS if k != "RematSplit"]))
MAXLEN.to_csv(os.path.join(RES_DIR, "max_seq_len.csv"))
print(MAXLEN.xs(1, level="link_scale").fillna(0).astype(int).to_string())

ml = MAXLEN.xs(1, level="link_scale")
gain_store = (ml["RematSplit"] / ml["Store (SplitLoRA)"].replace(0, np.nan)).groupby(level="device").mean()
gain_ckpt = (ml["RematSplit"] / ml["Checkpoint all"].replace(0, np.nan)).groupby(level="device").mean()
sub = SWEEP[(SWEEP.L == L_FIG) & (SWEEP.M_GB == M_FIG_GB) & SWEEP.feasible]
e = sub.pivot_table(index=["device", "link_scale"], columns="method", values="energy")
sav = {f"energy_saving_vs_{b.split(' (')[0].lower().replace(' ', '_')}_%": (1 - e["RematSplit"] / e[b]) * 100
       for b in ["Checkpoint all", "Greedy", "Offload all"] if b in e}
summary = pd.DataFrame(sav)
summary["max_len_gain_vs_store"] = summary.index.get_level_values("device").map(gain_store)
summary["max_len_gain_vs_checkpoint"] = summary.index.get_level_values("device").map(gain_ckpt)
summary.to_csv(os.path.join(RES_DIR, "headline_numbers.csv"))
print("\nHeadline numbers (use these in the abstract):\n", summary.round(2).to_string())

## 11. Experiment 3: online policy on dynamic 5G link (Hedge regret)

In [ ]:
devO = DEVICES[ONLINE_DEVICE]
L_on = max([l for l in L_OK if l <= L_ONLINE])
M_on = M_ONLINE_GB * GB
T_on = min(ONLINE_STEPS, len(TRACE))
QS = np.linspace(0.05, 0.95, 10)

experts = []
for q in QS:
    for obj in ("energy", "time"):
        p = dp_plan(L_on, M_on, devO, link_static(ONLINE_SCALE, q), objective=obj)
        if p is not None and p not in experts: experts.append(p)
for p in [dp_plan(L_on, M_on, devO, link_static(ONLINE_SCALE, 0.5), modes=ms) for ms in (["S", "R", "Q"], ["S", "R", "T"], ["S", "R", "TQ"], ["R", "TQ"])] \
         + [greedy_plan(L_on, M_on), ["R"] * len(UNITS), ["TQ"] * len(UNITS), ["T"] * len(UNITS)]:
    if p is not None and feasible(p, L_on, M_on) and p not in experts: experts.append(p)
Kx = len(experts)
assert Kx > 0, "no feasible plan at this L and memory budget; lower L_ONLINE or raise M_ONLINE_GB"

C = np.zeros((T_on, Kx)); oracle = np.zeros(T_on); ewma_c = np.zeros(T_on)
static_plan = dp_plan(L_on, M_on, devO, link_static(ONLINE_SCALE, 0.5))
static_c = np.zeros(T_on)
est, alpha = None, 0.3
for t in range(T_on):
    lk = link_at(t, ONLINE_SCALE)
    for j, p in enumerate(experts):
        C[t, j] = evaluate(p, L_on, M_on, devO, lk)["energy"]
    oracle[t] = evaluate(dp_plan(L_on, M_on, devO, lk), L_on, M_on, devO, lk)["energy"]
    static_c[t] = evaluate(static_plan, L_on, M_on, devO, lk)["energy"]
    pe_ = dp_plan(L_on, M_on, devO, est) if est else static_plan
    ewma_c[t] = evaluate(pe_, L_on, M_on, devO, lk)["energy"]
    est = dict(lk) if est is None else {k: alpha * lk[k] + (1 - alpha) * est[k] for k in lk}

Cmax = C.max()
eta = math.sqrt(8 * math.log(max(Kx, 2)) / T_on)
w = np.ones(Kx) / Kx; hedge = np.zeros(T_on)
for t in range(T_on):
    hedge[t] = w @ C[t]
    w = w * np.exp(-eta * C[t] / Cmax); w /= w.sum()

best_fixed = C.sum(0).min()
ONLINE = pd.DataFrame({"t": np.arange(T_on), "dl_mbps": TRACE.dl_mbps.values[:T_on],
                       "oracle": oracle, "hedge": hedge, "ewma": ewma_c, "static": static_c,
                       "best_fixed": C[:, C.sum(0).argmin()]})
for c in ["hedge", "ewma", "static"]:
    ONLINE[f"regret_{c}"] = np.cumsum(ONLINE[c] - ONLINE["best_fixed"])
    ONLINE[f"gap_oracle_{c}"] = np.cumsum(ONLINE[c] - ONLINE["oracle"])
ONLINE["regret_bound"] = Cmax * np.sqrt((np.arange(T_on) + 1) * math.log(max(Kx, 2)) / 2)
ONLINE.to_csv(os.path.join(RES_DIR, "online_policy.csv"), index=False)
print(f"{ONLINE_DEVICE}, link x{ONLINE_SCALE}, L={L_on}, M={M_ONLINE_GB} GB, experts={Kx}, steps={T_on}")
print(ONLINE[["oracle", "hedge", "ewma", "static", "best_fixed"]].sum().rename("total energy (J)").round(1).to_string())
print(f"Hedge regret vs best fixed plan: {ONLINE.regret_hedge.iloc[-1]:.1f} J "
      f"(bound {ONLINE.regret_bound.iloc[-1]:.1f} J)")

## 12. Experiment 4: real fine-tuning runs and memory model validation

In [ ]:
devJ, linkJ = DEVICES["Jetson Orin"], link_static(1)
Lt = TRAIN_SEQ
pk_store = plan_peak(["S"] * len(UNITS), Lt); pk_ckpt = plan_peak(["R"] * len(UNITS), Lt)
M_train = (DEV_WEIGHT_BYTES + 2 * smashed_bytes(Lt) + RUNTIME_OVERHEAD) + 0.5 * (pk_store + pk_ckpt)
TRAIN_PLANS = {"Store (SplitLoRA)": ["S"] * len(UNITS), "Checkpoint all": ["R"] * len(UNITS),
               "RematSplit": dp_plan(Lt, M_train, devJ, linkJ)}
print("RematSplit training plan:", "".join(TRAIN_PLANS["RematSplit"]), f"(budget {M_train/GB:.2f} GB)")

@torch.no_grad()
def eval_ppl():
    tot = 0.0
    for j in range(EVAL_SAMPLES):
        ids, lab = get_sample(Lt, j * 3, pool=HELD_IDX)
        smashed, pe, _ = device_forward(ids, ["S"] * len(UNITS))
        tot += float(server_loss(smashed, lab, pe))
    return math.exp(tot / EVAL_SAMPLES)

train_rows, ppl_rows = [], []
for name, plan in TRAIN_PLANS.items():
    reset_lora(); seed_all()
    opt = torch.optim.AdamW(trainable(), lr=LR, weight_decay=0.0)
    ppl0 = eval_ppl()
    for step in range(TRAIN_STEPS):
        ids, lab = get_sample(Lt, step * 2)
        out = split_step(ids, lab, plan, opt)
        train_rows.append(dict(method=name, step=step, **out))
        if step % 50 == 0: print(f"{name:20s} step {step:4d} loss {out['loss']:.4f}")
    ppl_rows.append(dict(method=name, ppl_before=ppl0, ppl_after=eval_ppl()))
    del opt; model.zero_grad(set_to_none=True); free()
reset_lora()
TRAIN = pd.DataFrame(train_rows); PPL = pd.DataFrame(ppl_rows)
TRAIN.to_csv(os.path.join(RES_DIR, "training_log.csv"), index=False)
PPL.to_csv(os.path.join(RES_DIR, "perplexity.csv"), index=False)
print(PPL.round(3).to_string(index=False))

val_rows = []
Mv = M_FIG_GB * GB
for L in L_OK:
    plan = dp_plan(L, Mv, devJ, linkJ)
    if plan is None: continue
    try:
        ids, _ = get_sample(L, 11)
        model.zero_grad(set_to_none=True); free()
        base_mem = mem_now(); reset_peak()
        smashed, pe, acc = device_forward(ids, plan)
        smashed.backward(torch.randn_like(smashed))
        meas = peak() - base_mem
        val_rows.append(dict(L=L, plan=",".join(plan), predicted_peak_GB=(plan_peak(plan, L) + 2 * smashed_bytes(L)) / GB,
                             measured_peak_GB=meas / GB, predicted_resident_GB=sum(P[(L, k, m)].res for k, m in zip(KINDS, plan)) / GB,
                             measured_resident_GB=acc.dev / GB, transmitted_GB=acc.tx / GB))
        del smashed, pe, acc
    except torch.cuda.OutOfMemoryError:
        print(f"validation OOM at L={L}")
    model.zero_grad(set_to_none=True); free()
VAL = pd.DataFrame(val_rows)
VAL.to_csv(os.path.join(RES_DIR, "memory_validation.csv"), index=False)
print(VAL.round(3).to_string(index=False))

## 13. Figures (each cell is self contained and reads the saved CSV files)

In [ ]:
import os, glob, numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib import font_manager as fm
for _f in glob.glob("/usr/share/fonts/truetype/msttcorefonts/*.ttf"): fm.fontManager.addfont(_f)
_FONT = "Times New Roman" if any(f.name == "Times New Roman" for f in fm.fontManager.ttflist) else "Liberation Serif"
plt.rcParams.update({"font.family": "serif", "font.serif": [_FONT, "DejaVu Serif"], "mathtext.fontset": "stix",
                     "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7,
                     "legend.fontsize": 7, "axes.linewidth": 0.6, "lines.linewidth": 1.1, "lines.markersize": 3.5,
                     "xtick.direction": "in", "ytick.direction": "in", "savefig.dpi": 800, "savefig.bbox": "tight",
                     "savefig.pad_inches": 0.02})
OUT_DIR = "/content/Downloads/RematSplit" if os.path.isdir("/content") else os.path.abspath("Downloads/RematSplit")
RES_DIR = os.path.join(OUT_DIR, "results")
COL = {"Store (SplitLoRA)": "#0072B2", "Checkpoint all": "#E69F00", "Offload all": "#009E73", "Greedy": "#CC79A7",
       "RematSplit": "#D55E00", "w/o latency hiding": "#56B4E9", "w/o transmit": "#F0E442", "w/o int8": "#999999",
       "Store/Recompute only": "#000000", "hedge": "#D55E00", "ewma": "#0072B2", "static": "#009E73"}
MK = {"Store (SplitLoRA)": "o", "Checkpoint all": "s", "Offload all": "^", "Greedy": "D", "RematSplit": "*"}
BASE_METHODS = ["Store (SplitLoRA)", "Checkpoint all", "Offload all", "Greedy", "RematSplit"]

S = __import__("json").load(open(os.path.join(RES_DIR, "settings.json")))
d = pd.read_csv(os.path.join(RES_DIR, "max_seq_len.csv"))
d = d[d.link_scale == 1]
devs = list(dict.fromkeys(d.device))
fig, axes = plt.subplots(1, len(devs), figsize=(7.16, 2.3), sharey=True)
w = 0.8 / len(BASE_METHODS)
for ax, dv in zip(np.atleast_1d(axes), devs):
    s = d[d.device == dv].set_index("M_GB")
    x = np.arange(len(s))
    for j, m in enumerate(BASE_METHODS):
        ax.bar(x + (j - (len(BASE_METHODS) - 1) / 2) * w, s[m].fillna(0).values / 1024, w * 0.92,
               color=COL[m], label=m, edgecolor="none")
    ax.set_xticks(x); ax.set_xticklabels([f"{v:g}" for v in s.index])
    ax.set_xlabel(f"Device memory budget (GB), {dv}")
    ax.set_yscale("log", base=2)
    ax.grid(axis="y", lw=0.3, alpha=0.5)
np.atleast_1d(axes)[0].set_ylabel("Max. sequence length (K tokens)")
h, l = np.atleast_1d(axes)[0].get_legend_handles_labels()
fig.tight_layout(rect=[0, 0.1, 1, 1])
fig.legend(h, l, loc="lower center", ncol=len(BASE_METHODS), frameon=False, bbox_to_anchor=(0.5, 0.0))
fig.savefig(os.path.join(OUT_DIR, "fig_max_seqlen.png")); plt.show()

In [ ]:
import os, glob, numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib import font_manager as fm
for _f in glob.glob("/usr/share/fonts/truetype/msttcorefonts/*.ttf"): fm.fontManager.addfont(_f)
_FONT = "Times New Roman" if any(f.name == "Times New Roman" for f in fm.fontManager.ttflist) else "Liberation Serif"
plt.rcParams.update({"font.family": "serif", "font.serif": [_FONT, "DejaVu Serif"], "mathtext.fontset": "stix",
                     "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7,
                     "legend.fontsize": 7, "axes.linewidth": 0.6, "lines.linewidth": 1.1, "lines.markersize": 3.5,
                     "xtick.direction": "in", "ytick.direction": "in", "savefig.dpi": 800, "savefig.bbox": "tight",
                     "savefig.pad_inches": 0.02})
OUT_DIR = "/content/Downloads/RematSplit" if os.path.isdir("/content") else os.path.abspath("Downloads/RematSplit")
RES_DIR = os.path.join(OUT_DIR, "results")
COL = {"Store (SplitLoRA)": "#0072B2", "Checkpoint all": "#E69F00", "Offload all": "#009E73", "Greedy": "#CC79A7",
       "RematSplit": "#D55E00", "w/o latency hiding": "#56B4E9", "w/o transmit": "#F0E442", "w/o int8": "#999999",
       "Store/Recompute only": "#000000", "hedge": "#D55E00", "ewma": "#0072B2", "static": "#009E73"}
MK = {"Store (SplitLoRA)": "o", "Checkpoint all": "s", "Offload all": "^", "Greedy": "D", "RematSplit": "*"}
BASE_METHODS = ["Store (SplitLoRA)", "Checkpoint all", "Offload all", "Greedy", "RematSplit"]

S = __import__("json").load(open(os.path.join(RES_DIR, "settings.json")))
d = pd.read_csv(os.path.join(RES_DIR, "plan_sweep.csv"))
d = d[(d.link_scale == S["LINK_FIG"]) & (d.M_GB == S["M_FIG_GB"]) & d.feasible & d.method.isin(BASE_METHODS)]
devs = list(dict.fromkeys(d.device))
fig, axes = plt.subplots(1, len(devs), figsize=(7.16, 2.3))
for ax, dv in zip(np.atleast_1d(axes), devs):
    for m in BASE_METHODS:
        s = d[(d.device == dv) & (d.method == m)].sort_values("L")
        if len(s): ax.plot(s.L / 1024, s.energy / 1e3, marker=MK[m], color=COL[m], label=m)
    ax.set_xscale("log", base=2); ax.set_yscale("log")
    ax.set_xlabel(f"Sequence length (K tokens), {dv}"); ax.set_ylabel("Energy per step (kJ)")
    ax.grid(lw=0.3, alpha=0.5)
h, l = np.atleast_1d(axes)[0].get_legend_handles_labels()
fig.tight_layout(rect=[0, 0.1, 1, 1])
fig.legend(h, l, loc="lower center", ncol=len(BASE_METHODS), frameon=False, bbox_to_anchor=(0.5, 0.0))
fig.savefig(os.path.join(OUT_DIR, "fig_energy_vs_len.png")); plt.show()

In [ ]:
import os, glob, numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib import font_manager as fm
for _f in glob.glob("/usr/share/fonts/truetype/msttcorefonts/*.ttf"): fm.fontManager.addfont(_f)
_FONT = "Times New Roman" if any(f.name == "Times New Roman" for f in fm.fontManager.ttflist) else "Liberation Serif"
plt.rcParams.update({"font.family": "serif", "font.serif": [_FONT, "DejaVu Serif"], "mathtext.fontset": "stix",
                     "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7,
                     "legend.fontsize": 7, "axes.linewidth": 0.6, "lines.linewidth": 1.1, "lines.markersize": 3.5,
                     "xtick.direction": "in", "ytick.direction": "in", "savefig.dpi": 800, "savefig.bbox": "tight",
                     "savefig.pad_inches": 0.02})
OUT_DIR = "/content/Downloads/RematSplit" if os.path.isdir("/content") else os.path.abspath("Downloads/RematSplit")
RES_DIR = os.path.join(OUT_DIR, "results")
COL = {"Store (SplitLoRA)": "#0072B2", "Checkpoint all": "#E69F00", "Offload all": "#009E73", "Greedy": "#CC79A7",
       "RematSplit": "#D55E00", "w/o latency hiding": "#56B4E9", "w/o transmit": "#F0E442", "w/o int8": "#999999",
       "Store/Recompute only": "#000000", "hedge": "#D55E00", "ewma": "#0072B2", "static": "#009E73"}
MK = {"Store (SplitLoRA)": "o", "Checkpoint all": "s", "Offload all": "^", "Greedy": "D", "RematSplit": "*"}
BASE_METHODS = ["Store (SplitLoRA)", "Checkpoint all", "Offload all", "Greedy", "RematSplit"]

S = __import__("json").load(open(os.path.join(RES_DIR, "settings.json")))
d = pd.read_csv(os.path.join(RES_DIR, "plan_sweep.csv"))
d = d[(d.link_scale == S["LINK_FIG"]) & (d.M_GB == S["M_FIG_GB"]) & (d.L == S["L_FIG"]) & d.feasible
      & (d.device == "Jetson Orin") & d.method.isin(BASE_METHODS)].set_index("method").reindex(BASE_METHODS).dropna(subset=["time"])
parts = [("t_comp", "Compute", "#0072B2"), ("t_recomp", "Recompute", "#E69F00"), ("t_link", "Link", "#009E73"), ("t_wait", "Wait", "#999999")]
fig, ax = plt.subplots(figsize=(3.5, 2.4))
x = np.arange(len(d)); bot = np.zeros(len(d))
for c, lab, col in parts:
    ax.bar(x, d[c].values, 0.6, bottom=bot, color=col, label=lab, edgecolor="none"); bot += d[c].values
ax.set_xticks(x); ax.set_xticklabels([m.replace(" (SplitLoRA)", "") for m in d.index], rotation=20, ha="right")
ax.set_yscale("log"); ax.set_ylabel("Step time (s)")
ax.set_xlabel(f"Method (L = {S['L_FIG']}, M = {S['M_FIG_GB']} GB, link x{S['LINK_FIG']}, Jetson Orin)")
ax.grid(axis="y", lw=0.3, alpha=0.5)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.42), ncol=4, frameon=False)
fig.savefig(os.path.join(OUT_DIR, "fig_time_breakdown.png")); plt.show()

In [ ]:
import os, glob, numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib import font_manager as fm
for _f in glob.glob("/usr/share/fonts/truetype/msttcorefonts/*.ttf"): fm.fontManager.addfont(_f)
_FONT = "Times New Roman" if any(f.name == "Times New Roman" for f in fm.fontManager.ttflist) else "Liberation Serif"
plt.rcParams.update({"font.family": "serif", "font.serif": [_FONT, "DejaVu Serif"], "mathtext.fontset": "stix",
                     "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7,
                     "legend.fontsize": 7, "axes.linewidth": 0.6, "lines.linewidth": 1.1, "lines.markersize": 3.5,
                     "xtick.direction": "in", "ytick.direction": "in", "savefig.dpi": 800, "savefig.bbox": "tight",
                     "savefig.pad_inches": 0.02})
OUT_DIR = "/content/Downloads/RematSplit" if os.path.isdir("/content") else os.path.abspath("Downloads/RematSplit")
RES_DIR = os.path.join(OUT_DIR, "results")
COL = {"Store (SplitLoRA)": "#0072B2", "Checkpoint all": "#E69F00", "Offload all": "#009E73", "Greedy": "#CC79A7",
       "RematSplit": "#D55E00", "w/o latency hiding": "#56B4E9", "w/o transmit": "#F0E442", "w/o int8": "#999999",
       "Store/Recompute only": "#000000", "hedge": "#D55E00", "ewma": "#0072B2", "static": "#009E73"}
MK = {"Store (SplitLoRA)": "o", "Checkpoint all": "s", "Offload all": "^", "Greedy": "D", "RematSplit": "*"}
BASE_METHODS = ["Store (SplitLoRA)", "Checkpoint all", "Offload all", "Greedy", "RematSplit"]

d = pd.read_csv(os.path.join(RES_DIR, "online_policy.csv"))
fig, ax = plt.subplots(figsize=(3.5, 2.4))
for c, lab in [("hedge", "RematSplit online (Hedge)"), ("ewma", "EWMA replanning"), ("static", "Static DP plan")]:
    ax.plot(d.t, d[f"regret_{c}"] / 1e3, color=COL[c], label=lab)
ax.plot(d.t, d.regret_bound / 1e3, color="k", ls="--", lw=0.8, label="Regret bound")
ax.set_xlabel("Training step (one 5G trace second per step)"); ax.set_ylabel("Cumulative regret (kJ)")
ax.grid(lw=0.3, alpha=0.5)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2, frameon=False)
fig.savefig(os.path.join(OUT_DIR, "fig_online_regret.png")); plt.show()

In [ ]:
import os, glob, numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib import font_manager as fm
for _f in glob.glob("/usr/share/fonts/truetype/msttcorefonts/*.ttf"): fm.fontManager.addfont(_f)
_FONT = "Times New Roman" if any(f.name == "Times New Roman" for f in fm.fontManager.ttflist) else "Liberation Serif"
plt.rcParams.update({"font.family": "serif", "font.serif": [_FONT, "DejaVu Serif"], "mathtext.fontset": "stix",
                     "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7,
                     "legend.fontsize": 7, "axes.linewidth": 0.6, "lines.linewidth": 1.1, "lines.markersize": 3.5,
                     "xtick.direction": "in", "ytick.direction": "in", "savefig.dpi": 800, "savefig.bbox": "tight",
                     "savefig.pad_inches": 0.02})
OUT_DIR = "/content/Downloads/RematSplit" if os.path.isdir("/content") else os.path.abspath("Downloads/RematSplit")
RES_DIR = os.path.join(OUT_DIR, "results")
COL = {"Store (SplitLoRA)": "#0072B2", "Checkpoint all": "#E69F00", "Offload all": "#009E73", "Greedy": "#CC79A7",
       "RematSplit": "#D55E00", "w/o latency hiding": "#56B4E9", "w/o transmit": "#F0E442", "w/o int8": "#999999",
       "Store/Recompute only": "#000000", "hedge": "#D55E00", "ewma": "#0072B2", "static": "#009E73"}
MK = {"Store (SplitLoRA)": "o", "Checkpoint all": "s", "Offload all": "^", "Greedy": "D", "RematSplit": "*"}
BASE_METHODS = ["Store (SplitLoRA)", "Checkpoint all", "Offload all", "Greedy", "RematSplit"]

d = pd.read_csv(os.path.join(RES_DIR, "training_log.csv"))
fig, ax = plt.subplots(figsize=(3.5, 2.4))
ls = {"Store (SplitLoRA)": "-", "Checkpoint all": "--", "RematSplit": ":"}
for m, s in d.groupby("method", sort=False):
    ax.plot(s.step, s.loss.rolling(10, min_periods=1).mean(), color=COL[m], ls=ls.get(m, "-"), label=m)
ax.set_xlabel("Training step"); ax.set_ylabel("Training loss (10 step mean)")
ax.grid(lw=0.3, alpha=0.5)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3, frameon=False)
fig.savefig(os.path.join(OUT_DIR, "fig_loss_curves.png")); plt.show()

In [ ]:
import os, glob, numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib import font_manager as fm
for _f in glob.glob("/usr/share/fonts/truetype/msttcorefonts/*.ttf"): fm.fontManager.addfont(_f)
_FONT = "Times New Roman" if any(f.name == "Times New Roman" for f in fm.fontManager.ttflist) else "Liberation Serif"
plt.rcParams.update({"font.family": "serif", "font.serif": [_FONT, "DejaVu Serif"], "mathtext.fontset": "stix",
                     "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7,
                     "legend.fontsize": 7, "axes.linewidth": 0.6, "lines.linewidth": 1.1, "lines.markersize": 3.5,
                     "xtick.direction": "in", "ytick.direction": "in", "savefig.dpi": 800, "savefig.bbox": "tight",
                     "savefig.pad_inches": 0.02})
OUT_DIR = "/content/Downloads/RematSplit" if os.path.isdir("/content") else os.path.abspath("Downloads/RematSplit")
RES_DIR = os.path.join(OUT_DIR, "results")
COL = {"Store (SplitLoRA)": "#0072B2", "Checkpoint all": "#E69F00", "Offload all": "#009E73", "Greedy": "#CC79A7",
       "RematSplit": "#D55E00", "w/o latency hiding": "#56B4E9", "w/o transmit": "#F0E442", "w/o int8": "#999999",
       "Store/Recompute only": "#000000", "hedge": "#D55E00", "ewma": "#0072B2", "static": "#009E73"}
MK = {"Store (SplitLoRA)": "o", "Checkpoint all": "s", "Offload all": "^", "Greedy": "D", "RematSplit": "*"}
BASE_METHODS = ["Store (SplitLoRA)", "Checkpoint all", "Offload all", "Greedy", "RematSplit"]

from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
S = __import__("json").load(open(os.path.join(RES_DIR, "settings.json")))
d = pd.read_csv(os.path.join(RES_DIR, "plan_sweep.csv"))
d = d[(d.link_scale == S["LINK_FIG"]) & (d.L == S["L_FIG"]) & (d.method == "RematSplit") & (d.device == "Jetson Orin") & d.feasible]
modes = ["S", "R", "T", "Q", "TQ"]; names = ["Store", "Recompute", "Transmit", "Int8 store", "Int8 transmit"]
cols = ["#0072B2", "#E69F00", "#009E73", "#56B4E9", "#CC79A7"]
grid = np.array([[modes.index(m) for m in p.split(",")] for p in d.sort_values("M_GB").plan])
fig, ax = plt.subplots(figsize=(3.5, 1.9))
ax.imshow(grid, cmap=ListedColormap(cols), vmin=-0.5, vmax=4.5, aspect="auto")
n = grid.shape[1]
ax.set_xticks(np.arange(0, n, 2)); ax.set_xticklabels([str(i + 1) for i in range(n // 2)])
ax.set_yticks(np.arange(len(d))); ax.set_yticklabels([f"{v:g} GB" for v in sorted(d.M_GB)])
ax.set_xlabel(f"Device decoder layer (attention, MLP units), L = {S['L_FIG']}"); ax.set_ylabel("Budget")
ax.legend(handles=[Patch(color=c, label=l) for c, l in zip(cols, names)], loc="upper center",
          bbox_to_anchor=(0.5, -0.27), ncol=5, frameon=False, handlelength=1.0, columnspacing=0.8)
fig.savefig(os.path.join(OUT_DIR, "fig_plan_modes.png")); plt.show()

In [ ]:
import os, glob, numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib import font_manager as fm
for _f in glob.glob("/usr/share/fonts/truetype/msttcorefonts/*.ttf"): fm.fontManager.addfont(_f)
_FONT = "Times New Roman" if any(f.name == "Times New Roman" for f in fm.fontManager.ttflist) else "Liberation Serif"
plt.rcParams.update({"font.family": "serif", "font.serif": [_FONT, "DejaVu Serif"], "mathtext.fontset": "stix",
                     "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7,
                     "legend.fontsize": 7, "axes.linewidth": 0.6, "lines.linewidth": 1.1, "lines.markersize": 3.5,
                     "xtick.direction": "in", "ytick.direction": "in", "savefig.dpi": 800, "savefig.bbox": "tight",
                     "savefig.pad_inches": 0.02})
OUT_DIR = "/content/Downloads/RematSplit" if os.path.isdir("/content") else os.path.abspath("Downloads/RematSplit")
RES_DIR = os.path.join(OUT_DIR, "results")
COL = {"Store (SplitLoRA)": "#0072B2", "Checkpoint all": "#E69F00", "Offload all": "#009E73", "Greedy": "#CC79A7",
       "RematSplit": "#D55E00", "w/o latency hiding": "#56B4E9", "w/o transmit": "#F0E442", "w/o int8": "#999999",
       "Store/Recompute only": "#000000", "hedge": "#D55E00", "ewma": "#0072B2", "static": "#009E73"}
MK = {"Store (SplitLoRA)": "o", "Checkpoint all": "s", "Offload all": "^", "Greedy": "D", "RematSplit": "*"}
BASE_METHODS = ["Store (SplitLoRA)", "Checkpoint all", "Offload all", "Greedy", "RematSplit"]

S = __import__("json").load(open(os.path.join(RES_DIR, "settings.json")))
d = pd.read_csv(os.path.join(RES_DIR, "plan_sweep.csv"))
abl = ["RematSplit", "w/o latency hiding", "w/o transmit", "w/o int8", "Store/Recompute only"]
d = d[(d.link_scale == S["LINK_FIG"]) & (d.L == S["L_FIG"]) & (d.M_GB == S["M_FIG_GB"]) & d.method.isin(abl) & d.feasible]
devs = list(dict.fromkeys(d.device))
fig, ax = plt.subplots(figsize=(3.5, 2.4))
w = 0.8 / len(abl); x = np.arange(len(devs))
for j, m in enumerate(abl):
    v = [d[(d.device == dv) & (d.method == m)].energy.mean() / d[(d.device == dv) & (d.method == "RematSplit")].energy.mean()
         for dv in devs]
    ax.bar(x + (j - (len(abl) - 1) / 2) * w, v, w * 0.92, color=COL[m], label=m, edgecolor="none")
ax.set_xticks(x); ax.set_xticklabels(devs)
ax.set_ylabel("Energy per step (normalised)"); ax.set_xlabel(f"Device (L = {S['L_FIG']}, M = {S['M_FIG_GB']} GB)")
ax.grid(axis="y", lw=0.3, alpha=0.5)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=3, frameon=False)
fig.savefig(os.path.join(OUT_DIR, "fig_ablation.png")); plt.show()

In [ ]:
import os, glob, numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib import font_manager as fm
for _f in glob.glob("/usr/share/fonts/truetype/msttcorefonts/*.ttf"): fm.fontManager.addfont(_f)
_FONT = "Times New Roman" if any(f.name == "Times New Roman" for f in fm.fontManager.ttflist) else "Liberation Serif"
plt.rcParams.update({"font.family": "serif", "font.serif": [_FONT, "DejaVu Serif"], "mathtext.fontset": "stix",
                     "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7,
                     "legend.fontsize": 7, "axes.linewidth": 0.6, "lines.linewidth": 1.1, "lines.markersize": 3.5,
                     "xtick.direction": "in", "ytick.direction": "in", "savefig.dpi": 800, "savefig.bbox": "tight",
                     "savefig.pad_inches": 0.02})
OUT_DIR = "/content/Downloads/RematSplit" if os.path.isdir("/content") else os.path.abspath("Downloads/RematSplit")
RES_DIR = os.path.join(OUT_DIR, "results")
COL = {"Store (SplitLoRA)": "#0072B2", "Checkpoint all": "#E69F00", "Offload all": "#009E73", "Greedy": "#CC79A7",
       "RematSplit": "#D55E00", "w/o latency hiding": "#56B4E9", "w/o transmit": "#F0E442", "w/o int8": "#999999",
       "Store/Recompute only": "#000000", "hedge": "#D55E00", "ewma": "#0072B2", "static": "#009E73"}
MK = {"Store (SplitLoRA)": "o", "Checkpoint all": "s", "Offload all": "^", "Greedy": "D", "RematSplit": "*"}
BASE_METHODS = ["Store (SplitLoRA)", "Checkpoint all", "Offload all", "Greedy", "RematSplit"]

d = pd.read_csv(os.path.join(RES_DIR, "memory_validation.csv")).sort_values("L")
fig, ax = plt.subplots(figsize=(3.5, 2.4))
ax.plot(d.L / 1024, d.predicted_peak_GB, marker="o", color="#0072B2", label="Predicted peak")
ax.plot(d.L / 1024, d.measured_peak_GB, marker="s", ls="--", color="#D55E00", label="Measured peak")
ax.set_xscale("log", base=2); ax.set_xlabel("Sequence length (K tokens)")
ax.set_ylabel("Device activation memory (GB)"); ax.grid(lw=0.3, alpha=0.5)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2, frameon=False)
fig.savefig(os.path.join(OUT_DIR, "fig_memory_validation.png")); plt.show()

In [ ]:
import os, glob, numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib import font_manager as fm
for _f in glob.glob("/usr/share/fonts/truetype/msttcorefonts/*.ttf"): fm.fontManager.addfont(_f)
_FONT = "Times New Roman" if any(f.name == "Times New Roman" for f in fm.fontManager.ttflist) else "Liberation Serif"
plt.rcParams.update({"font.family": "serif", "font.serif": [_FONT, "DejaVu Serif"], "mathtext.fontset": "stix",
                     "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7,
                     "legend.fontsize": 7, "axes.linewidth": 0.6, "lines.linewidth": 1.1, "lines.markersize": 3.5,
                     "xtick.direction": "in", "ytick.direction": "in", "savefig.dpi": 800, "savefig.bbox": "tight",
                     "savefig.pad_inches": 0.02})
OUT_DIR = "/content/Downloads/RematSplit" if os.path.isdir("/content") else os.path.abspath("Downloads/RematSplit")
RES_DIR = os.path.join(OUT_DIR, "results")
COL = {"Store (SplitLoRA)": "#0072B2", "Checkpoint all": "#E69F00", "Offload all": "#009E73", "Greedy": "#CC79A7",
       "RematSplit": "#D55E00", "w/o latency hiding": "#56B4E9", "w/o transmit": "#F0E442", "w/o int8": "#999999",
       "Store/Recompute only": "#000000", "hedge": "#D55E00", "ewma": "#0072B2", "static": "#009E73"}
MK = {"Store (SplitLoRA)": "o", "Checkpoint all": "s", "Offload all": "^", "Greedy": "D", "RematSplit": "*"}
BASE_METHODS = ["Store (SplitLoRA)", "Checkpoint all", "Offload all", "Greedy", "RematSplit"]

S = __import__("json").load(open(os.path.join(RES_DIR, "settings.json")))
d = pd.read_csv(os.path.join(RES_DIR, "plan_sweep.csv"))
d = d[(d.L == S["L_FIG"]) & (d.M_GB == S["M_FIG_GB"]) & (d.device == "Jetson Orin") & d.feasible & d.method.isin(BASE_METHODS)]
fig, ax = plt.subplots(figsize=(3.5, 2.4))
for m in BASE_METHODS:
    s = d[d.method == m].sort_values("link_scale")
    if len(s): ax.plot(s.link_scale, s.energy / 1e3, marker=MK[m], color=COL[m], label=m)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Link capacity scale (relative to 5G trace)"); ax.set_ylabel("Energy per step (kJ)")
ax.grid(lw=0.3, alpha=0.5)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3, frameon=False)
fig.savefig(os.path.join(OUT_DIR, "fig_link_sensitivity.png")); plt.show()

## 14. Download all results

In [ ]:
import os, shutil
OUT_DIR = "/content/Downloads/RematSplit" if os.path.isdir("/content") else os.path.abspath("Downloads/RematSplit")
zip_path = shutil.make_archive("/content/RematSplit_results" if os.path.isdir("/content") else "RematSplit_results", "zip", OUT_DIR)
try:
    from google.colab import files; files.download(zip_path)
except ImportError:
    print("Saved:", zip_path)